# Notebook 6: Spread & Total (O/U) Projections
## Project point spreads and game totals for tournament matchups
---
**Models:** Linear Regression, Random Forest, XGBoost, Bayesian Linear Regression  
**Output:** Matchup table with: favorite win %, spread, each team's implied points, game total

**Usage:** Run for each round. Update `TARGET_ROUND` in cell 6.0 as the tournament progresses.

**Input:** `data/features_2026.csv`, `data/historical_features.csv`, `data/historical_tourney.csv`, `data/bracket_2026.csv`

In [1]:
# ============================================================
# 6.0 CONFIG & IMPORTS
# ============================================================
import os, json, pickle, warnings, logging
import numpy as np
import pandas as pd
from pathlib import Path
from datetime import datetime

warnings.filterwarnings("ignore")
logging.basicConfig(level=logging.INFO, format="%(asctime)s [%(levelname)s] %(message)s")
log = logging.getLogger("spreads")

DATA_DIR = Path("./data")
MODEL_DIR = Path("./models")
RESULTS_DIR = Path("./results")

RANDOM_SEED = 51
SALT = 2026_03
SEED = RANDOM_SEED + SALT
np.random.seed(SEED)

# Which round to project? Update as tournament progresses.
TARGET_ROUND = "R64"  # Options: R64, R32, S16, E8, F4, Championship

log.info(f"Projecting spreads & totals for: {TARGET_ROUND}")

2026-03-18 23:40:14,863 [INFO] Projecting spreads & totals for: R64


In [2]:
# ============================================================
# 6.1 LOAD DATA & BUILD TRAINING SET
# ============================================================
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer

hist_tourney = pd.read_csv(DATA_DIR / "historical_tourney.csv")
hist_features = pd.read_csv(DATA_DIR / "historical_features.csv")
features_2026 = pd.read_csv(DATA_DIR / "features_2026.csv")
bracket_2026 = pd.read_csv(DATA_DIR / "bracket_2026.csv")

# Merge score data
hist_tourney["total_points"] = hist_tourney["score1"] + hist_tourney["score2"]
hist_merged = hist_features.merge(
    hist_tourney[["season", "team1", "team2", "total_points", "score1", "score2"]],
    on=["season", "team1", "team2"], how="left"
)
log.info(f"Historical matchups with scores: {hist_merged['total_points'].notna().sum()}/{len(hist_merged)}")

# --- SPREAD FEATURES (differential — predict who wins and by how much) ---
SPREAD_FEATURES = [
    "seed_diff", "seed_sum", "abs_seed_diff", "log_seed_ratio",
    "SRS_diff", "SOS_diff", "eFG%_diff", "TOV%_diff",
    "ORtg_diff", "Pace_diff", "ORB%_diff",
    "AdjOE_diff", "AdjDE_diff", "Barthag_diff",
]

# --- TOTAL FEATURES (level/sum — predict combined scoring pace) ---
TOTAL_FEATURES = SPREAD_FEATURES + [
    "Pace_avg", "ORtg_avg", "SRS_sum",
    "AdjOE_avg", "AdjDE_avg", "AdjTempo_avg",
    "OE_vs_DE_1", "OE_vs_DE_2",
]

spread_feats = [f for f in SPREAD_FEATURES if f in hist_merged.columns]
total_feats = [f for f in TOTAL_FEATURES if f in hist_merged.columns]
log.info(f"Spread features: {len(spread_feats)}")
log.info(f"Total features: {len(total_feats)}")

# Separate masks for spread vs total (total needs level features)
mask_spread = hist_merged["score_diff"].notna()
for f in spread_feats:
    mask_spread &= hist_merged[f].notna()

mask_total = hist_merged["total_points"].notna()
for f in total_feats:
    mask_total &= hist_merged[f].notna()

train_spread = hist_merged[mask_spread].copy()
train_total = hist_merged[mask_total].copy()

X_spread = train_spread[spread_feats].values
y_spread = train_spread["score_diff"].values

X_total = train_total[total_feats].values
y_total = train_total["total_points"].values

# Imputers and scalers (separate for spread vs total since different feature sets)
imp_spread = SimpleImputer(strategy="median")
sc_spread = StandardScaler()
X_spread_sc = sc_spread.fit_transform(imp_spread.fit_transform(X_spread))

imp_total = SimpleImputer(strategy="median")
sc_total = StandardScaler()
X_total_sc = sc_total.fit_transform(imp_total.fit_transform(X_total))

log.info(f"Spread training: {X_spread_sc.shape[0]} games × {X_spread_sc.shape[1]} features")
log.info(f"Total training: {X_total_sc.shape[0]} games × {X_total_sc.shape[1]} features")
log.info(f"Spread stats: mean={y_spread.mean():.1f}, std={y_spread.std():.1f}")
log.info(f"Total stats: mean={y_total.mean():.1f}, std={y_total.std():.1f}, "
         f"range=[{y_total.min():.0f}, {y_total.max():.0f}]")

2026-03-18 23:40:15,605 [INFO] Historical matchups with scores: 1448/1448
2026-03-18 23:40:15,606 [INFO] Spread features: 14
2026-03-18 23:40:15,606 [INFO] Total features: 22
2026-03-18 23:40:15,611 [INFO] Spread training: 878 games × 14 features
2026-03-18 23:40:15,611 [INFO] Total training: 878 games × 22 features
2026-03-18 23:40:15,612 [INFO] Spread stats: mean=4.9, std=13.9
2026-03-18 23:40:15,612 [INFO] Total stats: mean=138.7, std=18.7, range=[90, 205]


In [3]:
# ============================================================
# 6.2 TRAIN SPREAD MODELS
# ============================================================
from sklearn.linear_model import LinearRegression, RidgeCV
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

log.info("Training spread models...")

# --- Linear Regression ---
lr_spread = RidgeCV(alphas=np.logspace(-3, 3, 20), cv=5)
lr_spread.fit(X_spread_sc, y_spread)
lr_spread_pred = lr_spread.predict(X_spread_sc)
log.info(f"  Ridge Spread - MAE: {mean_absolute_error(y_spread, lr_spread_pred):.2f}, "
         f"R²: {r2_score(y_spread, lr_spread_pred):.3f}")

# --- Random Forest ---
rf_spread = RandomForestRegressor(
    n_estimators=500, max_depth=6, min_samples_leaf=10,
    random_state=SEED, n_jobs=-1
)
rf_spread.fit(X_spread_sc, y_spread)
rf_spread_pred = rf_spread.predict(X_spread_sc)
log.info(f"  RF Spread - MAE: {mean_absolute_error(y_spread, rf_spread_pred):.2f}, "
         f"R²: {r2_score(y_spread, rf_spread_pred):.3f}")

# --- XGBoost ---
from xgboost import XGBRegressor
xgb_spread = XGBRegressor(
    n_estimators=300, max_depth=4, learning_rate=0.05,
    subsample=0.8, colsample_bytree=0.8,
    random_state=SEED, n_jobs=-1
)
xgb_spread.fit(X_spread_sc, y_spread)
xgb_spread_pred = xgb_spread.predict(X_spread_sc)
log.info(f"  XGB Spread - MAE: {mean_absolute_error(y_spread, xgb_spread_pred):.2f}, "
         f"R²: {r2_score(y_spread, xgb_spread_pred):.3f}")

# --- Bayesian Ridge ---
from sklearn.linear_model import BayesianRidge
bayes_spread = BayesianRidge()
bayes_spread.fit(X_spread_sc, y_spread)
bayes_spread_pred, bayes_spread_std = bayes_spread.predict(X_spread_sc, return_std=True)
log.info(f"  Bayesian Spread - MAE: {mean_absolute_error(y_spread, bayes_spread_pred):.2f}, "
         f"R²: {r2_score(y_spread, bayes_spread_pred):.3f}, "
         f"Mean uncertainty: ±{bayes_spread_std.mean():.1f}")

2026-03-18 23:40:15,668 [INFO] Training spread models...
2026-03-18 23:40:15,706 [INFO]   Ridge Spread - MAE: 7.99, R²: 0.469
2026-03-18 23:40:15,991 [INFO]   RF Spread - MAE: 7.03, R²: 0.592
2026-03-18 23:40:16,260 [INFO]   XGB Spread - MAE: 3.83, R²: 0.876
2026-03-18 23:40:16,262 [INFO]   Bayesian Spread - MAE: 8.00, R²: 0.468, Mean uncertainty: ±10.3


In [4]:
# ============================================================
# 6.3 TRAIN TOTAL MODELS
# ============================================================
log.info("Training total (O/U) models...")

lr_total = RidgeCV(alphas=np.logspace(-3, 3, 20), cv=5)
lr_total.fit(X_total_sc, y_total)
lr_total_pred = lr_total.predict(X_total_sc)
log.info(f"  Ridge Total - MAE: {mean_absolute_error(y_total, lr_total_pred):.2f}, "
         f"R²: {r2_score(y_total, lr_total_pred):.3f}")

rf_total = RandomForestRegressor(
    n_estimators=500, max_depth=6, min_samples_leaf=10,
    random_state=SEED, n_jobs=-1
)
rf_total.fit(X_total_sc, y_total)
rf_total_pred = rf_total.predict(X_total_sc)
log.info(f"  RF Total - MAE: {mean_absolute_error(y_total, rf_total_pred):.2f}, "
         f"R²: {r2_score(y_total, rf_total_pred):.3f}")

xgb_total = XGBRegressor(
    n_estimators=300, max_depth=4, learning_rate=0.05,
    subsample=0.8, colsample_bytree=0.8,
    random_state=SEED, n_jobs=-1
)
xgb_total.fit(X_total_sc, y_total)
xgb_total_pred = xgb_total.predict(X_total_sc)
log.info(f"  XGB Total - MAE: {mean_absolute_error(y_total, xgb_total_pred):.2f}, "
         f"R²: {r2_score(y_total, xgb_total_pred):.3f}")

bayes_total = BayesianRidge()
bayes_total.fit(X_total_sc, y_total)
bayes_total_pred, bayes_total_std = bayes_total.predict(X_total_sc, return_std=True)
log.info(f"  Bayesian Total - MAE: {mean_absolute_error(y_total, bayes_total_pred):.2f}, "
         f"R²: {r2_score(y_total, bayes_total_pred):.3f}, "
         f"Mean uncertainty: ±{bayes_total_std.mean():.1f}")

2026-03-18 23:40:16,265 [INFO] Training total (O/U) models...
2026-03-18 23:40:16,305 [INFO]   Ridge Total - MAE: 12.03, R²: 0.328
2026-03-18 23:40:16,653 [INFO]   RF Total - MAE: 10.16, R²: 0.512
2026-03-18 23:40:16,913 [INFO]   XGB Total - MAE: 4.82, R²: 0.889
2026-03-18 23:40:16,915 [INFO]   Bayesian Total - MAE: 12.02, R²: 0.330, Mean uncertainty: ±15.5


In [5]:
# ============================================================
# 6.4 PROJECT CURRENT ROUND MATCHUPS
# ============================================================
def build_matchup_features_2026(team1_name, team2_name):
    """Build feature vector for a 2026 matchup using features_2026.csv."""
    t1 = features_2026[features_2026["team"] == team1_name]
    t2 = features_2026[features_2026["team"] == team2_name]
    
    if len(t1) == 0 or len(t2) == 0:
        log.warning(f"  Team not found: {team1_name if len(t1)==0 else team2_name}")
        return None
    
    t1, t2 = t1.iloc[0], t2.iloc[0]
    
    seed1, seed2 = t1["seed"], t2["seed"]
    seed_diff = seed1 - seed2
    
    features = {
        "seed_diff": seed_diff,
        "seed_sum": seed1 + seed2,
        "abs_seed_diff": abs(seed_diff),
        "log_seed_ratio": np.log(max(seed2, 0.5) / max(seed1, 0.5)),
        "SRS_diff": safe_diff(t1, t2, "SRS"),
        "SOS_diff": safe_diff(t1, t2, "SOS"),
        "eFG%_diff": safe_diff(t1, t2, "eFG%"),
        "TOV%_diff": safe_diff(t1, t2, "TOV%"),
        "ORtg_diff": safe_diff(t1, t2, "ORtg"),
        "Pace_diff": safe_diff(t1, t2, "Pace"),
        "ORB%_diff": safe_diff(t1, t2, "ORB%"),
    }
    return np.array([[features.get(c, np.nan) for c in available_feats]])


def safe_diff(t1, t2, col):
    try:
        return float(t1[col]) - float(t2[col])
    except (KeyError, TypeError, ValueError):
        return np.nan


# Build R64 matchups from bracket
bracket_order = [1, 16, 8, 9, 5, 12, 4, 13, 6, 11, 3, 14, 7, 10, 2, 15]
region_order = ["East", "West", "South", "Midwest"]

# Only generate for target round
if TARGET_ROUND == "R64":
    matchups = []
    for region in region_order:
        reg = bracket_2026[bracket_2026["region"] == region].set_index("seed")
        for i in range(0, len(bracket_order), 2):
            s1, s2 = bracket_order[i], bracket_order[i+1]
            if s1 in reg.index and s2 in reg.index:
                t1_name = reg.loc[s1, "team"]
                t2_name = reg.loc[s2, "team"]
                matchups.append({
                    "region": region, "team1": t1_name, "seed1": s1,
                    "team2": t2_name, "seed2": s2,
                })
    log.info(f"Built {len(matchups)} Round of 64 matchups")
else:
    # For later rounds: load simulation results and pick most likely matchups
    sim = pd.read_csv(RESULTS_DIR / "simulation_results.csv")
    log.info(f"For {TARGET_ROUND}: manually update matchups list based on actual results")
    matchups = []  # User fills in actual matchups as tournament progresses

print(f"\nProjecting {len(matchups)} matchups for {TARGET_ROUND}")

2026-03-18 23:40:16,920 [INFO] Built 32 Round of 64 matchups



Projecting 32 matchups for R64


In [6]:
# ============================================================
# 6.4 BUILD 2026 MATCHUP FEATURES
# ============================================================
def build_spread_features_2026(t1_name, t2_name):
    """Build spread (differential) features for a 2026 matchup."""
    t1 = features_2026[features_2026["team"] == t1_name]
    t2 = features_2026[features_2026["team"] == t2_name]
    if len(t1) == 0 or len(t2) == 0:
        return None
    t1, t2 = t1.iloc[0], t2.iloc[0]
    
    features = {}
    s1, s2 = t1["seed"], t2["seed"]
    sd = s1 - s2
    features["seed_diff"] = sd
    features["seed_sum"] = s1 + s2
    features["abs_seed_diff"] = abs(sd)
    features["log_seed_ratio"] = np.log(max(s2, 0.5) / max(s1, 0.5))
    
    diff_pairs = {
        "SRS_diff": "SRS", "SOS_diff": "SOS", "eFG%_diff": "eFG%",
        "TOV%_diff": "TOV%", "ORtg_diff": "ORtg", "Pace_diff": "Pace",
        "ORB%_diff": "ORB%", "AdjOE_diff": "bt_AdjOE",
        "AdjDE_diff": "bt_AdjDE", "Barthag_diff": "bt_Barthag",
    }
    for feat, col in diff_pairs.items():
        try:
            features[feat] = float(t1[col]) - float(t2[col])
        except:
            features[feat] = np.nan
    
    return np.array([[features.get(c, np.nan) for c in spread_feats]])


def build_total_features_2026(t1_name, t2_name):
    """Build total (level + differential) features for a 2026 matchup."""
    t1 = features_2026[features_2026["team"] == t1_name]
    t2 = features_2026[features_2026["team"] == t2_name]
    if len(t1) == 0 or len(t2) == 0:
        return None
    t1, t2 = t1.iloc[0], t2.iloc[0]
    
    # Start with all spread features
    features = {}
    s1, s2 = t1["seed"], t2["seed"]
    sd = s1 - s2
    features["seed_diff"] = sd
    features["seed_sum"] = s1 + s2
    features["abs_seed_diff"] = abs(sd)
    features["log_seed_ratio"] = np.log(max(s2, 0.5) / max(s1, 0.5))
    
    diff_pairs = {
        "SRS_diff": "SRS", "SOS_diff": "SOS", "eFG%_diff": "eFG%",
        "TOV%_diff": "TOV%", "ORtg_diff": "ORtg", "Pace_diff": "Pace",
        "ORB%_diff": "ORB%", "AdjOE_diff": "bt_AdjOE",
        "AdjDE_diff": "bt_AdjDE", "Barthag_diff": "bt_Barthag",
    }
    for feat, col in diff_pairs.items():
        try: features[feat] = float(t1[col]) - float(t2[col])
        except: features[feat] = np.nan
    
    # Level features — these drive total prediction
    try: features["Pace_avg"] = (float(t1["Pace"]) + float(t2["Pace"])) / 2
    except: features["Pace_avg"] = np.nan
    try: features["ORtg_avg"] = (float(t1["ORtg"]) + float(t2["ORtg"])) / 2
    except: features["ORtg_avg"] = np.nan
    try: features["SRS_sum"] = float(t1["SRS"]) + float(t2["SRS"])
    except: features["SRS_sum"] = np.nan
    try: features["AdjOE_avg"] = (float(t1["bt_AdjOE"]) + float(t2["bt_AdjOE"])) / 2
    except: features["AdjOE_avg"] = np.nan
    try: features["AdjDE_avg"] = (float(t1["bt_AdjDE"]) + float(t2["bt_AdjDE"])) / 2
    except: features["AdjDE_avg"] = np.nan
    try: features["AdjTempo_avg"] = (float(t1["bt_AdjTempo"]) + float(t2["bt_AdjTempo"])) / 2
    except: features["AdjTempo_avg"] = np.nan
    # Cross terms: each team's offense vs opponent's defense
    try: features["OE_vs_DE_1"] = float(t1["bt_AdjOE"]) - float(t2["bt_AdjDE"])
    except: features["OE_vs_DE_1"] = np.nan
    try: features["OE_vs_DE_2"] = float(t2["bt_AdjOE"]) - float(t1["bt_AdjDE"])
    except: features["OE_vs_DE_2"] = np.nan
    
    return np.array([[features.get(c, np.nan) for c in total_feats]])


# Build R64 matchups from bracket
bracket_order = [1, 16, 8, 9, 5, 12, 4, 13, 6, 11, 3, 14, 7, 10, 2, 15]
region_order = ["East", "West", "South", "Midwest"]

matchups = []
if TARGET_ROUND == "R64":
    for region in region_order:
        reg = bracket_2026[bracket_2026["region"] == region].set_index("seed")
        for i in range(0, len(bracket_order), 2):
            s1, s2 = bracket_order[i], bracket_order[i+1]
            if s1 in reg.index and s2 in reg.index:
                matchups.append({
                    "region": region, "team1": reg.loc[s1, "team"], "seed1": s1,
                    "team2": reg.loc[s2, "team"], "seed2": s2,
                })
    log.info(f"Built {len(matchups)} Round of 64 matchups")
else:
    log.info(f"For {TARGET_ROUND}: manually update matchups list")

print(f"\nProjecting {len(matchups)} matchups for {TARGET_ROUND}")

2026-03-18 23:40:16,928 [INFO] Built 32 Round of 64 matchups



Projecting 32 matchups for R64


In [8]:
# ============================================================
# 6.5 GENERATE PROJECTIONS
# ============================================================
sim_results = pd.read_csv(RESULTS_DIR / "simulation_results.csv")

projections = []
for m in matchups:
    t1, t2 = m["team1"], m["team2"]
    s1, s2 = m["seed1"], m["seed2"]
    region = m["region"]
    
    # --- Spread prediction ---
    X_sp = build_spread_features_2026(t1, t2)
    if X_sp is None:
        continue
    X_sp_sc = sc_spread.transform(imp_spread.transform(X_sp))
    
    spreads = [
        lr_spread.predict(X_sp_sc)[0],
        rf_spread.predict(X_sp_sc)[0],
        xgb_spread.predict(X_sp_sc)[0],
        bayes_spread.predict(X_sp_sc)[0],
    ]
    spread_mean = np.mean(spreads)
    
    # --- Total prediction ---
    X_tot = build_total_features_2026(t1, t2)
    if X_tot is None:
        continue
    X_tot_sc = sc_total.transform(imp_total.transform(X_tot))
    
    totals = [
        lr_total.predict(X_tot_sc)[0],
        rf_total.predict(X_tot_sc)[0],
        xgb_total.predict(X_tot_sc)[0],
        bayes_total.predict(X_tot_sc)[0],
    ]
    total_mean = np.mean(totals)
    
    # --- Model agreement (useful signal: when models disagree, be cautious) ---
    spread_range = max(spreads) - min(spreads)
    total_range = max(totals) - min(totals)
    
    # Win probability from simulation
    t1_sim = sim_results[sim_results["team"] == t1]
    t2_sim = sim_results[sim_results["team"] == t2]
    t1_r32 = t1_sim["prob_R32"].values[0] if len(t1_sim) > 0 else 0.5
    t2_r32 = t2_sim["prob_R32"].values[0] if len(t2_sim) > 0 else 0.5
    t1_win_prob = t1_r32 / (t1_r32 + t2_r32) if (t1_r32 + t2_r32) > 0 else 0.5
    
    # Determine favorite
    if spread_mean > 0:
        fav, dog = t1, t2
        fav_seed, dog_seed = s1, s2
        fav_prob = t1_win_prob
    else:
        fav, dog = t2, t1
        fav_seed, dog_seed = s2, s1
        fav_prob = 1 - t1_win_prob
    
    spread_line = -abs(spread_mean)
    fav_pts = (total_mean / 2) + (abs(spread_mean) / 2)
    dog_pts = (total_mean / 2) - (abs(spread_mean) / 2)
    
    projections.append({
        "Region": region,
        "Favorite": f"({fav_seed}) {fav}",
        "Underdog": f"({dog_seed}) {dog}",
        "Fav Win%": f"{fav_prob:.1%}",
        "Spread": round(spread_mean if spread_mean < 0 else -spread_mean, 1),
        "Total": round(total_mean, 1),
        "Fav Pts": round(fav_pts, 1),
        "Dog Pts": round(dog_pts, 1),
        "Spread Range": round(spread_range, 1),
        "Total Range": round(total_range, 1),
    })

proj_df = pd.DataFrame(projections)
proj_df.to_csv(RESULTS_DIR / f"projections_{TARGET_ROUND}.csv", index=False)

print(f"\n{TARGET_ROUND} PROJECTIONS")
print("=" * 110)
display(proj_df)


R64 PROJECTIONS


,Region,Favorite,Underdog,Fav Win%,Spread,Total,Fav Pts,Dog Pts,Spread Range,Total Range
0,East,(1) Duke,(16) Siena,97.3%,-30.5,142.3,86.4,55.9,10.2,3.9
1,East,(8) Ohio St.,(9) TCU,80.9%,-8.6,153.7,81.1,72.5,1.8,10.1
2,East,(5) St. John's,(12) Northern Iowa,83.2%,-11.3,139.6,75.5,64.2,4.0,7.3
3,East,(4) Kansas,(13) Cal Baptist,92.1%,-15.2,136.8,76.0,60.8,7.5,9.5
4,East,(6) Louisville,(11) South Florida,78.5%,-7.4,165.5,86.5,79.1,5.3,8.8
5,East,(3) Michigan St.,(14) North Dakota St.,94.9%,-20.1,146.9,83.5,63.4,3.5,3.2
6,East,(7) UCLA,(10) UCF,83.0%,-9.3,158.1,83.7,74.4,4.9,10.0
7,East,(2) UConn,(15) Furman,97.1%,-26.4,143.1,84.7,58.4,1.0,2.0
8,West,(1) Arizona,(16) Long Island,97.6%,-33.9,155.4,94.7,60.7,7.5,0.7
9,West,(9) Utah St.,(8) Villanova,44.1%,-2.8,148.3,75.5,72.7,5.4,10.2


## Spread & Total Projection Summary

### Models Used:
- **Ridge Regression** — regularized linear baseline
- **Random Forest** — non-linear interactions
- **XGBoost** — gradient boosted trees
- **Bayesian Ridge** — uncertainty quantification (±X points)

### How to Use for Betting:
1. Compare "Spread" to the sportsbook line → if model says -8.5 and book says -5.5, the model thinks the favorite is undervalued
2. Compare "Total" to the book's O/U → if model says 148 and book says 142, lean OVER
3. "Spread Unc" and "Total Unc" tell you confidence — larger uncertainty = less conviction
4. "Betting Value" section flags games where model disagrees most with seed-implied lines

### For Later Rounds:
Change `TARGET_ROUND` and manually enter matchups as results come in.